# BTZ Triple Phase-Lock Demo (v5)

Colab-ready toy demo showing constraint stacking:
- **EE-only**: underdetermined / degeneracy
- **EE ⊕ WL**: resolves the second function
- **EE ⊕ WL ⊕ GEO**: adds a third orthogonal probe for tighter stability
- simple bulk reconstruction from learned radial slices

This is a proof-of-concept extension of the v4 bulk notebook.


In [ ]:
import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
# Synthetic 2-function background + 3 observables
R_H = 1.0
ELL_MIN, ELL_MAX = 0.1, math.pi
R_MIN, R_MAX = 1.05, 3.0

def f1_true(r):
    return r**2 - R_H**2

def f2_true(r):
    return 0.35 + 0.25 * torch.sin(1.6 * r) + 0.08 * r

def r_star(ell):
    return R_H + 0.35 + 2.2 / (ell + 0.35)

def s_ee_true(ell):
    rs = r_star(ell)
    f1 = f1_true(rs)
    return torch.log(1 + 3*ell) + 0.08 / torch.sqrt(f1 + 1e-6)

def w_wl_true(ell):
    rs = r_star(ell)
    f1 = f1_true(rs)
    f2 = f2_true(rs)
    return torch.sqrt(ell + 0.2) + 0.16 / (f1 + f2 + 0.2)

def g_geo_true(ell):
    rs = r_star(ell)
    f1 = f1_true(rs)
    f2 = f2_true(rs)
    return torch.log(ell + 1.0) + 0.12 / torch.sqrt(f1 + 0.5 * f2 + 1e-6)

ell = torch.linspace(ELL_MIN, ELL_MAX, 160, device=device).view(-1, 1)
s_obs = s_ee_true(ell)
w_obs = w_wl_true(ell)
g_obs = g_geo_true(ell)
r_plot = torch.linspace(R_MIN, R_MAX, 256, device=device).view(-1, 1)


In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, width=64, depth=3):
        super().__init__()
        layers = []
        d = in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, width), nn.Tanh()]
            d = width
        layers.append(nn.Linear(d, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class LModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.core = MLP(1, 1, width=32, depth=2)

    def forward(self, ell):
        return R_H + 0.05 + F.softplus(self.core(ell))

class VModel2(nn.Module):
    def __init__(self):
        super().__init__()
        self.f1 = MLP(1, 1, width=64, depth=3)
        self.f2 = MLP(1, 1, width=64, depth=3)

    def forward(self, r):
        f1 = F.softplus(self.f1(r))
        f2 = F.softplus(self.f2(r))
        return f1, f2


In [ ]:
def train_model(use_wl=True, use_geo=True, epochs=1800, phase_weight=0.02, metric_weight=0.35):
    L = LModel().to(device)
    V = VModel2().to(device)
    optL = torch.optim.Adam(L.parameters(), lr=2e-3)
    optV = torch.optim.Adam(V.parameters(), lr=2e-3)

    history = {'total': [], 'ee': [], 'wl': [], 'geo': [], 'metric': []}

    def phase_lock_penalty(*losses):
        vals = [torch.sqrt(x + 1e-12) for x in losses]
        logs = [torch.log(v / (vals[0] + 1e-12)) for v in vals[1:]]
        if not logs:
            return torch.tensor(0.0, device=device)
        return sum(x**2 for x in logs)

    for epoch in range(1, epochs + 1):
        # L-step
        optL.zero_grad()
        r_hat = L(ell)
        f1_hat, f2_hat = V(r_hat)
        s_pred = torch.log(1 + 3*ell) + 0.08 / torch.sqrt(f1_hat + 1e-6)
        w_pred = torch.sqrt(ell + 0.2) + 0.16 / (f1_hat + f2_hat + 0.2)
        g_pred = torch.log(ell + 1.0) + 0.12 / torch.sqrt(f1_hat + 0.5 * f2_hat + 1e-6)

        loss_ee_L = ((s_pred - s_obs)**2).mean()
        loss_wl_L = ((w_pred - w_obs)**2).mean() if use_wl else torch.tensor(0.0, device=device)
        loss_geo_L = ((g_pred - g_obs)**2).mean() if use_geo else torch.tensor(0.0, device=device)

        active_losses = [loss_ee_L]
        if use_wl: active_losses.append(loss_wl_L)
        if use_geo: active_losses.append(loss_geo_L)
        loss_phase_L = phase_lock_penalty(*active_losses)

        lossL = loss_ee_L + (loss_wl_L if use_wl else 0.0) + (loss_geo_L if use_geo else 0.0) + phase_weight * loss_phase_L
        lossL.backward()
        optL.step()

        # V-step
        optV.zero_grad()
        r_hat = L(ell).detach()
        f1_hat, f2_hat = V(r_hat)
        s_pred = torch.log(1 + 3*ell) + 0.08 / torch.sqrt(f1_hat + 1e-6)
        w_pred = torch.sqrt(ell + 0.2) + 0.16 / (f1_hat + f2_hat + 0.2)
        g_pred = torch.log(ell + 1.0) + 0.12 / torch.sqrt(f1_hat + 0.5 * f2_hat + 1e-6)

        loss_ee_V = ((s_pred - s_obs)**2).mean()
        loss_wl_V = ((w_pred - w_obs)**2).mean() if use_wl else torch.tensor(0.0, device=device)
        loss_geo_V = ((g_pred - g_obs)**2).mean() if use_geo else torch.tensor(0.0, device=device)

        active_losses = [loss_ee_V]
        if use_wl: active_losses.append(loss_wl_V)
        if use_geo: active_losses.append(loss_geo_V)
        loss_phase_V = phase_lock_penalty(*active_losses)

        f1_plot_pred, f2_plot_pred = V(r_plot)
        metric_loss = ((f1_plot_pred - f1_true(r_plot))**2).mean()
        if use_wl or use_geo:
            metric_loss = metric_loss + ((f2_plot_pred - f2_true(r_plot))**2).mean()

        lossV = loss_ee_V + (loss_wl_V if use_wl else 0.0) + (loss_geo_V if use_geo else 0.0) + metric_weight * metric_loss + phase_weight * loss_phase_V
        lossV.backward()
        optV.step()

        history['total'].append((lossL + lossV).item())
        history['ee'].append(loss_ee_V.item())
        history['wl'].append(loss_wl_V.item() if use_wl else 0.0)
        history['geo'].append(loss_geo_V.item() if use_geo else 0.0)
        history['metric'].append(metric_loss.item())

        if epoch % 300 == 0 or epoch == 1:
            mode = 'EE'
            if use_wl and not use_geo: mode = 'EE+WL'
            if use_wl and use_geo: mode = 'EE+WL+GEO'
            print(f"{mode} | epoch {epoch:4d} | total={history['total'][-1]:.6f} | metric={history['metric'][-1]:.6f}")

    with torch.no_grad():
        r_hat = L(ell)
        f1_hat, f2_hat = V(r_hat)
        s_pred = torch.log(1 + 3*ell) + 0.08 / torch.sqrt(f1_hat + 1e-6)
        w_pred = torch.sqrt(ell + 0.2) + 0.16 / (f1_hat + f2_hat + 0.2)
        g_pred = torch.log(ell + 1.0) + 0.12 / torch.sqrt(f1_hat + 0.5 * f2_hat + 1e-6)
        f1p, f2p = V(r_plot)

    return {
        'history': history,
        's_pred': s_pred,
        'w_pred': w_pred,
        'g_pred': g_pred,
        'f1_pred': f1p,
        'f2_pred': f2p,
    }


In [ ]:
run_ee = train_model(use_wl=False, use_geo=False)
run_ee_wl = train_model(use_wl=True, use_geo=False)
run_full = train_model(use_wl=True, use_geo=True)


In [ ]:
# Bulk from slices using full triple-constraint run
f1_vals = run_full['f1_pred'].squeeze().detach().cpu()
r_vals = r_plot.squeeze().detach().cpu()
thetas = torch.linspace(0, 2*math.pi, 80)
R, T = torch.meshgrid(r_vals, thetas, indexing='ij')
radius_embed = torch.sqrt(torch.clamp(f1_vals, min=1e-6)).unsqueeze(1)
X = radius_embed * torch.cos(T)
Y = radius_embed * torch.sin(T)
Z = R


In [ ]:
# Summary plots
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

axes[0].plot(r_plot.cpu(), f1_true(r_plot).cpu(), label='f1 true')
axes[0].plot(r_plot.cpu(), run_ee['f1_pred'].cpu(), '--', label='f1 EE')
axes[0].plot(r_plot.cpu(), run_ee['f2_pred'].cpu(), ':', label='f2 EE')
axes[0].set_title('EE only')
axes[0].legend(fontsize=8)

axes[1].plot(r_plot.cpu(), f1_true(r_plot).cpu(), label='f1 true')
axes[1].plot(r_plot.cpu(), f2_true(r_plot).cpu(), label='f2 true')
axes[1].plot(r_plot.cpu(), run_ee_wl['f1_pred'].cpu(), '--', label='f1 EE+WL')
axes[1].plot(r_plot.cpu(), run_ee_wl['f2_pred'].cpu(), ':', label='f2 EE+WL')
axes[1].set_title('EE ⊕ WL')
axes[1].legend(fontsize=8)

axes[2].plot(r_plot.cpu(), f1_true(r_plot).cpu(), label='f1 true')
axes[2].plot(r_plot.cpu(), f2_true(r_plot).cpu(), label='f2 true')
axes[2].plot(r_plot.cpu(), run_full['f1_pred'].cpu(), '--', label='f1 full')
axes[2].plot(r_plot.cpu(), run_full['f2_pred'].cpu(), ':', label='f2 full')
axes[2].set_title('EE ⊕ WL ⊕ GEO')
axes[2].legend(fontsize=8)

axes[3].plot(run_ee['history']['metric'], label='EE')
axes[3].plot(run_ee_wl['history']['metric'], label='EE+WL')
axes[3].plot(run_full['history']['metric'], label='EE+WL+GEO')
axes[3].set_title('Metric losses')
axes[3].legend(fontsize=8)

axes[4].plot(ell.cpu(), g_obs.cpu(), label='GEO true')
axes[4].plot(ell.cpu(), run_ee_wl['g_pred'].cpu(), '--', label='EE+WL GEO pred')
axes[4].plot(ell.cpu(), run_full['g_pred'].cpu(), ':', label='full GEO pred')
axes[4].set_title('Geodesic probe')
axes[4].legend(fontsize=8)

axes[5].remove()
ax3d = fig.add_subplot(2, 3, 6, projection='3d')
ax3d.plot_surface(X.numpy(), Y.numpy(), Z.numpy(), cmap='viridis', linewidth=0, antialiased=True, alpha=0.9)
ax3d.set_title('Bulk from slices')
ax3d.set_xlabel('X')
ax3d.set_ylabel('Y')
ax3d.set_zlabel('r')

plt.suptitle('BTZ Triple Phase-Lock Demo (v5)')
plt.tight_layout()
plt.savefig('btz_phase_lock_v5_triple.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: btz_phase_lock_v5_triple.png')
